# 病例复杂度调用样例

本笔记本将演示：

1. 如何调用朝厚云 API 生成病例复杂度分析结果。

使用本笔记本前，您需要为病例准备以下材料：

1. 上下颌口腔扫描网格模型。

In [ ]:
# 导入必要的包以及定义函数
import os
import time
import requests
import json
import trimesh
import urllib
import numpy as np

# 定义调用规则

请根据您从我方获取的信息修改以下代码块

In [ ]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

In [ ]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    print(response.text)
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    print(response.text)
    return response.json()

def retrieve_data(urn):
    return requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

##  病例复杂度

该章节多个调用可以使用以下单一调用代替

```python
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "case-complexity-analysis", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id
}
```

### 获取分牙结果

In [ ]:
# 上颌
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-denoise-prod",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "mesh": {"type":"drc", "data": upload_file("upper_jaw_scan.drc")},
      "jaw_type": "Upper"
  },
  'output_config': {
    "teeth_comp": {"type": "ply"}
  }
}
result_upper_jaw = run_job_and_get_results(json_call, 300)

In [ ]:
# 下颌
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-denoise-prod",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "mesh": {"type":"ply", "data": upload_file("lower_jaw_scan.ply")},
      "jaw_type": "Lower"
  },
  'output_config': {
    "teeth_comp": {"type": "ply"}
  }
}
result_lower_jaw = run_job_and_get_results(json_call, 300)

### 获取目标定位结果

In [ ]:

json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "auto-arrange", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data":{
      "upper_teeth_dict": result_upper_jaw["teeth_comp"],
      "upper_axis_matrix_dict": result_upper_jaw["axis"],
      "lower_teeth_dict": result_lower_jaw["teeth_comp"],
      "lower_axis_matrix_dict": result_lower_jaw["axis"],

  },
}
# result_upper_jaw["teeth_comp"]
# result_upper_jaw["axis"]
# #result_lower_jaw["teeth_comp"]
# #result_lower_jaw["axis"]
result_arrangement = run_job_and_get_results(json_call, 500)

### 病例复杂度

In [ ]:
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "case-complexity-analysis", 
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data":{
      "teeth_dict": {**result_upper_jaw["teeth_comp"], **result_lower_jaw["teeth_comp"]},
      "axis_dict": {**result_upper_jaw["axis"], **result_lower_jaw["axis"]},
      "transformation_dict": result_arrangement["result"]["transformation_dict"],
      "landmarks_dict": {**result_upper_jaw["landmarks"], **result_lower_jaw["landmarks"]},
      "target_out_form": "" # 暂无对应功能，传入空字符串
  },

}
result_case_complexity= run_job_and_get_results(json_call, 500)

In [ ]:
print("=== Case Complexity Analysis Results ===")
print(json.dumps(result_case_complexity, indent=4, ensure_ascii=False))